# Stage 04 — DoubleML Extension

Estimate causal effects with Double/Debiased Machine Learning following
Baiardi & Naghi (2024). Run all 12 specifications (4 outcomes x 3 treatments)
with 6 ML learners + Ensemble + Best, median aggregation, and B&N adjusted SE.
Then compute BLP, GATE, CLAN heterogeneity analysis and Causal Forest.

In [1]:
from paths import *
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from scipy import stats
import statsmodels.api as sm

import doubleml as dml
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor
)
from sklearn.linear_model import LassoCV, ElasticNetCV
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler

# Causal forest imports
try:
    from econml.dml import CausalForestDML
    ECONML_AVAILABLE = True
except ImportError:
    ECONML_AVAILABLE = False
    print('WARNING: econml not installed. Causal Forest will be skipped.')

# Load data and specs
df = pd.read_parquet(str(DATASET_PATH))
spec = json.loads(PAPER_SPEC.read_text())
rep = json.loads(REPLICATION_RESULTS.read_text())

# Primary specification
outcome = spec['identification']['outcome_variable']
treatment = spec['identification']['treatment_variable']
controls = spec['identification']['controls']
id_type = spec['identification']['type']
model_type = spec['dml']['model_type']

# Full 4x3 grid
outcomes = [o['variable'] for o in spec['identification']['additional_outcomes']]
outcome_labels = {o['variable']: o['label'] for o in spec['identification']['additional_outcomes']}
treatments = [t['variable'] for t in spec['identification']['additional_treatments']]
treatment_labels = {t['variable']: t['label'] for t in spec['identification']['additional_treatments']}

panel_labels = {
    'Investment2005': 'Panel A: Investment',
    'FDI2005': 'Panel B: FDI',
    'nbusinessdensitycf': 'Panel C: Business density',
    'nentryrateav': 'Panel D: Entry rate'
}

# Adaptive K-folds: K=2 for N<200 (following B&N)
test_cols = [outcome, treatment] + controls
n_test = df[test_cols].dropna().shape[0]
N_FOLDS = 2 if n_test < 200 else 5
N_REP = 5  # acceptable for re-run (B&N use 100)

# 6 ML learners following B&N
ML_METHODS = ['Lasso', 'ElasticNet', 'DecisionTree', 'Boosting', 'Forest', 'NeuralNet']

print(f'DML model: {model_type}')
print(f'Outcomes ({len(outcomes)}): {outcomes}')
print(f'Treatments ({len(treatments)}): {treatments}')
print(f'Controls ({len(controls)}): {controls}')
print(f'ML methods: {ML_METHODS}')
print(f'Cross-fitting: K={N_FOLDS} folds (N={n_test}<200), n_rep={N_REP}')
print(f'econml available: {ECONML_AVAILABLE}')

DML model: PLR
Outcomes (4): ['Investment2005', 'FDI2005', 'nbusinessdensitycf', 'nentryrateav']
Treatments (3): ['statutory', 'effective', 'effective_5yr']
Controls (12): ['other_taxes', 'vatsales', 'pit2004', 'lnpayments2004', 'lngdppc2003', 'propertyrights', 'sb_proc2004', 'emplo_i2004', 'freedomtotradeinternationally', 'seign2004', 'inf10yearavg', 'index_gcr']
ML methods: ['Lasso', 'ElasticNet', 'DecisionTree', 'Boosting', 'Forest', 'NeuralNet']
Cross-fitting: K=2 folds (N=61<200), n_rep=5
econml available: False


In [2]:
# --- Helpers ---

def make_learner(method_name, n_obs, n_features):
    """Return a fresh sklearn regressor adapted to sample size."""
    if method_name == 'Lasso':
        return LassoCV(cv=min(5, max(2, n_obs // 10)), max_iter=10000, random_state=42)
    elif method_name == 'ElasticNet':
        return ElasticNetCV(
            cv=min(5, max(2, n_obs // 10)),
            l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.95, 0.99],
            max_iter=10000, random_state=42
        )
    elif method_name == 'DecisionTree':
        return DecisionTreeRegressor(
            ccp_alpha=0.01, max_depth=min(5, max(2, n_obs // 10)), random_state=42
        )
    elif method_name == 'Boosting':
        return GradientBoostingRegressor(
            n_estimators=500, learning_rate=0.01,
            max_depth=2, subsample=0.5,
            min_samples_leaf=max(1, n_obs // 20), random_state=42
        )
    elif method_name == 'Forest':
        return RandomForestRegressor(
            n_estimators=500, min_samples_leaf=5,
            max_features='sqrt', n_jobs=-1, random_state=42
        )
    elif method_name == 'NeuralNet':
        return MLPRegressor(
            hidden_layer_sizes=(64,), max_iter=1000,
            early_stopping=True, learning_rate_init=0.01, random_state=42
        )
    else:
        raise ValueError(f'Unknown method: {method_name}')

def get_ols_result(outcome_var, treatment_var):
    for s in rep['specs']:
        if s['outcome'] == outcome_var and s['treatment'] == treatment_var:
            return s
    return None

def extract_lasso_diagnostics(obj, controls_list, d_col):
    """Extract Lasso/ElasticNet variable selection info."""
    try:
        # Try different key formats across DoubleML versions
        for key_l in ['ml_l0', d_col, 'ml_l']:
            try:
                models_l = obj.models['ml_l'][key_l]
                model_l = models_l[0][0]
                break
            except (KeyError, TypeError):
                continue
        else:
            return None
        for key_m in ['ml_m0', d_col, 'ml_m']:
            try:
                models_m = obj.models['ml_m'][key_m]
                model_m = models_m[0][0]
                break
            except (KeyError, TypeError):
                continue
        else:
            return None

        coefs_l = pd.Series(model_l.coef_, index=controls_list)
        nonzero_l = coefs_l[coefs_l != 0].sort_values(key=abs, ascending=False)
        coefs_m = pd.Series(model_m.coef_, index=controls_list)
        nonzero_m = coefs_m[coefs_m != 0].sort_values(key=abs, ascending=False)

        return {
            'outcome_model': {
                'n_nonzero': int(len(nonzero_l)),
                'total_p': len(controls_list),
                'top_variables': {k: round(v, 4) for k, v in nonzero_l.head(5).items()},
            },
            'treatment_model': {
                'n_nonzero': int(len(nonzero_m)),
                'total_p': len(controls_list),
                'top_variables': {k: round(v, 4) for k, v in nonzero_m.head(5).items()},
            },
        }
    except Exception:
        return None

print('Helpers defined.')

Helpers defined.


In [3]:
# --- Run DML for all 12 specs (4 outcomes x 3 treatments) x 6 ML methods ---
# Median aggregation with B&N adjusted SE, adaptive K=2 folds

all_dml_results = []
fitted_objects = {}  # (outcome, treatment, method) -> fitted DoubleMLPLR

for y_var in outcomes:
    for d_var in treatments:
        key_cols = [y_var, d_var] + controls
        df_spec = df[key_cols].dropna().reset_index(drop=True)
        n_obs = len(df_spec)
        if n_obs < 10:
            print(f'  SKIP {y_var} ~ {d_var}: N={n_obs} too small')
            continue

        k_folds = 2 if n_obs < 200 else 5
        dml_data = dml.DoubleMLData(df_spec, y_col=y_var, d_cols=d_var, x_cols=controls)
        ols = get_ols_result(y_var, d_var)

        spec_result = {
            'outcome': {'variable': y_var, 'label': outcome_labels.get(y_var, y_var)},
            'treatment': {'variable': d_var, 'label': treatment_labels.get(d_var, d_var)},
            'n_obs': n_obs, 'n_folds': k_folds, 'n_rep': N_REP,
            'ols_coef': float(ols['replicated_coef']) if ols else None,
            'ols_se': float(ols['replicated_se']) if ols else None,
            'learners': {}
        }
        learner_nuisance_mse = {}
        print(f'\n=== {panel_labels.get(y_var, y_var)} ~ {d_var} (N={n_obs}, K={k_folds}) ===')

        for method_name in ML_METHODS:
            try:
                ml_l = make_learner(method_name, n_obs, len(controls))
                ml_m = make_learner(method_name, n_obs, len(controls))
                obj = dml.DoubleMLPLR(dml_data, ml_l=ml_l, ml_m=ml_m, n_folds=k_folds, n_rep=N_REP)
                obj.fit()
                fitted_objects[(y_var, d_var, method_name)] = obj

                # Median aggregation with B&N adjusted SE
                all_coefs = obj.all_coef.flatten()
                all_ses = obj.all_se.flatten()
                median_coef = float(np.median(all_coefs))
                se_adj = float(np.median(np.sqrt(all_ses**2 + (all_coefs - median_coef)**2)))
                ci_lo = median_coef - 1.96 * se_adj
                ci_hi = median_coef + 1.96 * se_adj
                pval = float(2 * (1 - stats.norm.cdf(abs(median_coef / se_adj)))) if se_adj > 0 else 1.0

                # Nuisance quality
                preds_y = obj.predictions['ml_l'][:, 0, 0]
                preds_d = obj.predictions['ml_m'][:, 0, 0]
                r2_y = float(r2_score(df_spec[y_var], preds_y))
                r2_d = float(r2_score(df_spec[d_var], preds_d))
                mse_y = float(mean_squared_error(df_spec[y_var], preds_y))
                mse_d = float(mean_squared_error(df_spec[d_var], preds_d))
                learner_nuisance_mse[method_name] = {'mse_outcome': mse_y, 'mse_treatment': mse_d, 'r2_outcome': r2_y, 'r2_treatment': r2_d}

                lasso_diag = None
                if method_name in ('Lasso', 'ElasticNet'):
                    lasso_diag = extract_lasso_diagnostics(obj, controls, d_var)

                sign_change = False
                if spec_result['ols_coef'] is not None and spec_result['ols_coef'] != 0:
                    sign_change = bool(np.sign(median_coef) != np.sign(spec_result['ols_coef']))

                spec_result['learners'][method_name] = {
                    'coef': median_coef, 'se': se_adj, 'ci_lo': ci_lo, 'ci_hi': ci_hi, 'pval': pval,
                    'per_rep_coefs': all_coefs.tolist(), 'per_rep_ses': all_ses.tolist(),
                    'nuisance': learner_nuisance_mse[method_name],
                    'lasso_diagnostics': lasso_diag, 'sign_change': sign_change,
                }
                stars = '***' if pval < 0.01 else ('**' if pval < 0.05 else ('*' if pval < 0.10 else ''))
                print(f'  {method_name:15s}: coef={median_coef:+.4f} SE={se_adj:.4f} p={pval:.3f}{stars}  R2(y)={r2_y:.3f} R2(d)={r2_d:.3f}')
            except Exception as e:
                print(f'  {method_name:15s}: FAILED - {e}')
                spec_result['learners'][method_name] = {'error': str(e)}

        # --- Ensemble: MSE-inverse-weighted ---
        methods_ok = [m for m in ML_METHODS if m in learner_nuisance_mse and 'coef' in spec_result['learners'].get(m, {})]
        if methods_ok:
            weights = {m: 1.0 / max(learner_nuisance_mse[m]['mse_outcome'], 1e-10) for m in methods_ok}
            total_w = sum(weights.values())
            weights = {k: v / total_w for k, v in weights.items()}
            ens_coef = sum(weights[m] * spec_result['learners'][m]['coef'] for m in methods_ok)
            ens_se = sum(weights[m] * spec_result['learners'][m]['se'] for m in methods_ok)
            ens_pval = float(2 * (1 - stats.norm.cdf(abs(ens_coef / ens_se)))) if ens_se > 0 else 1.0
            spec_result['learners']['Ensemble'] = {
                'coef': float(ens_coef), 'se': float(ens_se),
                'ci_lo': float(ens_coef - 1.96 * ens_se), 'ci_hi': float(ens_coef + 1.96 * ens_se),
                'pval': ens_pval, 'weights': {k: round(v, 4) for k, v in weights.items()},
            }
            stars = '***' if ens_pval < 0.01 else ('**' if ens_pval < 0.05 else ('*' if ens_pval < 0.10 else ''))
            print(f'  {"Ensemble":15s}: coef={ens_coef:+.4f} SE={ens_se:.4f} p={ens_pval:.3f}{stars}')

        # --- Best: lowest nuisance MSE (NOT p-value) ---
        if methods_ok:
            best_method = min(methods_ok, key=lambda m: learner_nuisance_mse[m]['mse_outcome'])
            best_treat_method = min(methods_ok, key=lambda m: learner_nuisance_mse[m]['mse_treatment'])
            best_lr = spec_result['learners'][best_method]
            spec_result['learners']['Best'] = {
                'coef': best_lr['coef'], 'se': best_lr['se'],
                'ci_lo': best_lr['ci_lo'], 'ci_hi': best_lr['ci_hi'], 'pval': best_lr['pval'],
                'best_outcome_method': best_method, 'best_treatment_method': best_treat_method,
                'selection_criterion': 'lowest_nuisance_MSE',
            }
            print(f'  {"Best":15s}: {best_method} (MSE_y={learner_nuisance_mse[best_method]["mse_outcome"]:.4f})')

        sign_change_pub = False
        if 'Best' in spec_result['learners'] and spec_result['ols_coef'] is not None and spec_result['ols_coef'] != 0:
            sign_change_pub = bool(np.sign(spec_result['learners']['Best']['coef']) != np.sign(spec_result['ols_coef']))

        spec_result['preferred_learner'] = 'Best'
        spec_result['preferred_coef'] = spec_result['learners'].get('Best', {}).get('coef')
        spec_result['preferred_ci_lo'] = spec_result['learners'].get('Best', {}).get('ci_lo')
        spec_result['preferred_ci_hi'] = spec_result['learners'].get('Best', {}).get('ci_hi')
        spec_result['sign_change_vs_published'] = sign_change_pub
        all_dml_results.append(spec_result)

print(f'\n=== Completed: {len(all_dml_results)} specifications ===')


=== Panel A: Investment ~ statutory (N=61, K=2) ===


  Lasso          : coef=-0.1112 SE=0.0668 p=0.096*  R2(y)=0.191 R2(d)=-0.028


  ElasticNet     : coef=-0.0948 SE=0.0879 p=0.281  R2(y)=0.079 R2(d)=-0.208
  DecisionTree   : coef=-0.0541 SE=0.1129 p=0.632  R2(y)=-0.582 R2(d)=-0.673


  Boosting       : coef=+0.0017 SE=0.0973 p=0.986  R2(y)=-0.371 R2(d)=0.206


  Forest         : coef=-0.0798 SE=0.0726 p=0.272  R2(y)=-0.130 R2(d)=0.020
  NeuralNet      : coef=+0.2032 SE=0.1426 p=0.154  R2(y)=-0.967 R2(d)=-0.504
  Ensemble       : coef=-0.0474 SE=0.0901 p=0.599
  Best           : Lasso (MSE_y=18.7257)

=== Panel A: Investment ~ effective (N=61, K=2) ===


...

  Forest         : coef=-0.1489 SE=0.0669 p=0.026**  R2(y)=0.068 R2(d)=0.098
  NeuralNet      : coef=-0.1971 SE=0.0789 p=0.012**  R2(y)=-0.312 R2(d)=-0.383
  Ensemble       : coef=-0.1324 SE=0.0678 p=0.051*
  Best           : Boosting (MSE_y=9.2873)

=== Panel D: Entry rate ~ effective_5yr (N=50, K=2) ===


  Lasso          : coef=-0.1226 SE=0.0649 p=0.059*  R2(y)=-0.238 R2(d)=-1.092


  ElasticNet     : coef=-0.1584 SE=0.0718 p=0.027**  R2(y)=-0.018 R2(d)=-0.330
  DecisionTree   : coef=-0.1147 SE=0.0688 p=0.095*  R2(y)=-0.221 R2(d)=-0.538


  Boosting       : coef=-0.1970 SE=0.0719 p=0.006***  R2(y)=-0.194 R2(d)=0.010


  Forest         : coef=-0.1744 SE=0.0733 p=0.017**  R2(y)=-0.241 R2(d)=0.090
  NeuralNet      : coef=-0.1059 SE=0.0560 p=0.059*  R2(y)=-0.117 R2(d)=-0.717
  Ensemble       : coef=-0.1455 SE=0.0677 p=0.032**
  Best           : ElasticNet (MSE_y=11.0948)

=== Completed: 12 specifications ===


In [4]:
# --- Write dml_results.json and compare with B&N reference ---

primary_spec = None
for s in all_dml_results:
    if s['outcome']['variable'] == outcome and s['treatment']['variable'] == treatment:
        primary_spec = s
        break

pref_lr = primary_spec['learners'].get('Best', {}) if primary_spec else {}

dml_output = {
    'model_type': model_type,
    'aggregation': 'median_across_reps',
    'se_formula': 'B&N adjusted: median(sqrt(SE_k^2 + (coef_k - median_coef)^2))',
    'n_folds': N_FOLDS, 'n_rep': N_REP,
    'ml_methods': ML_METHODS,
    'preferred_learner': 'Best',
    'preferred_coef': pref_lr.get('coef'),
    'preferred_se': pref_lr.get('se'),
    'preferred_ci_lo': pref_lr.get('ci_lo'),
    'preferred_ci_hi': pref_lr.get('ci_hi'),
    'preferred_pval': pref_lr.get('pval'),
    'specifications': all_dml_results,
}
DML_RESULTS.write_text(json.dumps(dml_output, indent=2, default=str))
print(f'Wrote {DML_RESULTS}')
print(f'  {len(all_dml_results)} specs x {len(ML_METHODS)} methods + Ensemble + Best')

# --- Compare with B&N Table 1 ---
bn_ref = {
    'Investment2005': {'Lasso': -0.178, 'Boosting': -0.199, 'Forest': -0.204, 'NNet': -0.218, 'Best': -0.203, 'OLS': -0.189},
    'FDI2005': {'Lasso': -0.162, 'Boosting': -0.169, 'Forest': -0.177, 'NNet': -0.164, 'Best': -0.15, 'OLS': -0.095},
    'nbusinessdensitycf': {'Lasso': -0.093, 'Boosting': -0.11, 'Forest': -0.104, 'NNet': -0.087, 'Best': -0.098, 'OLS': -0.070},
    'nentryrateav': {'Lasso': -0.156, 'Boosting': -0.155, 'Forest': -0.15, 'NNet': -0.175, 'Best': -0.152, 'OLS': -0.133},
}
method_map = {'Lasso': 'Lasso', 'Boosting': 'Boosting', 'Forest': 'Forest', 'NeuralNet': 'NNet'}

print('\n=== Comparison with B&N Table 1 (effective_5yr) ===')
print(f'{"Outcome":25s} {"Method":10s} {"Ours":>8s} {"B&N":>8s} {"Diff":>8s}')
print('-' * 65)
for s in all_dml_results:
    if s['treatment']['variable'] != 'effective_5yr':
        continue
    y = s['outcome']['variable']
    if y not in bn_ref:
        continue
    for our_m, bn_m in method_map.items():
        if our_m in s['learners'] and 'coef' in s['learners'][our_m]:
            our_c = s['learners'][our_m]['coef']
            bn_c = bn_ref[y].get(bn_m)
            if bn_c is not None:
                diff = our_c - bn_c
                print(f'{panel_labels.get(y, y):25s} {our_m:10s} {our_c:+8.3f} {bn_c:+8.3f} {diff:+8.3f}')

Wrote C:\Users\qgallea\Dropbox\work\claude_code\recast\papers\04_djankov_et_al_taxes\data\results\dml_results.json
  12 specs x 6 methods + Ensemble + Best

=== Comparison with B&N Table 1 (effective_5yr) ===
Outcome                   Method         Ours      B&N     Diff
-----------------------------------------------------------------
Panel A: Investment       Lasso        -0.080   -0.178   +0.098
Panel A: Investment       Boosting     -0.244   -0.199   -0.045
Panel A: Investment       Forest       -0.218   -0.204   -0.014
Panel A: Investment       NeuralNet    +0.072   -0.218   +0.290
Panel B: FDI              Lasso        -0.130   -0.162   +0.032
Panel B: FDI              Boosting     -0.149   -0.169   +0.020
Panel B: FDI              Forest       -0.187   -0.177   -0.010
Panel B: FDI              NeuralNet    -0.115   -0.164   +0.049
Panel C: Business density Lasso        -0.148   -0.093   -0.055
Panel C: Business density Boosting     -0.118   -0.110   -0.008
Panel C: Business den

In [5]:
# --- BLP Heterogeneity Test (primary specification) ---
# BLP: (Y - Y_hat) = b1*(D - D_hat) + b2*(D - D_hat)*(S(X) - mean(S)) + e
# b2 tests for systematic heterogeneity

primary_y = outcome
primary_d = treatment
key_cols_primary = [primary_y, primary_d] + controls
df_primary = df[key_cols_primary].dropna().reset_index(drop=True)
n_primary = len(df_primary)

best_method_primary = None
if primary_spec and 'Best' in primary_spec['learners']:
    best_method_primary = primary_spec['learners']['Best'].get('best_outcome_method', 'Forest')

obj_best = fitted_objects.get((primary_y, primary_d, best_method_primary))

blp_results = None
S_proxy = None

if obj_best is not None:
    preds_y = obj_best.predictions['ml_l'][:, 0, 0]
    preds_d = obj_best.predictions['ml_m'][:, 0, 0]
    Y_res = df_primary[primary_y].values - preds_y
    D_res = df_primary[primary_d].values - preds_d

    # CATE proxy S(X) = Y_res / D_res (regularized)
    d_res_safe = np.where(np.abs(D_res) < 0.01, np.nan, D_res)
    S_raw = Y_res / d_res_safe
    S_raw_clean = S_raw[~np.isnan(S_raw)]
    if len(S_raw_clean) > 5:
        lo_q, hi_q = np.nanpercentile(S_raw_clean, [2, 98])
        S_proxy = np.clip(np.where(np.isnan(S_raw), np.nanmedian(S_raw_clean), S_raw), lo_q, hi_q)
    else:
        S_proxy = Y_res

    S_centered = S_proxy - np.mean(S_proxy)
    X_blp = np.column_stack([D_res, D_res * S_centered])
    blp_model = sm.OLS(Y_res, X_blp).fit(cov_type='HC1')

    blp_results = {
        'beta1_ate': float(blp_model.params[0]),
        'beta1_se': float(blp_model.bse[0]),
        'beta1_pval': float(blp_model.pvalues[0]),
        'beta2_het': float(blp_model.params[1]),
        'beta2_se': float(blp_model.bse[1]),
        'beta2_pval': float(blp_model.pvalues[1]),
        'heterogeneity_significant': bool(blp_model.pvalues[1] < 0.10),
    }
    print('=== BLP Heterogeneity Test ===')
    print(f'  beta1 (ATE):           {blp_results["beta1_ate"]:.4f} (SE={blp_results["beta1_se"]:.4f}, p={blp_results["beta1_pval"]:.4f})')
    print(f'  beta2 (heterogeneity): {blp_results["beta2_het"]:.4f} (SE={blp_results["beta2_se"]:.4f}, p={blp_results["beta2_pval"]:.4f})')
    print(f'  Significant at 10%: {"YES" if blp_results["heterogeneity_significant"] else "NO"}')
else:
    print('WARNING: Could not find fitted Best model. Skipping BLP.')

=== BLP Heterogeneity Test ===
  beta1 (ATE):           -0.1554 (SE=0.0018, p=0.0000)
  beta2 (heterogeneity): 1.0194 (SE=0.0110, p=0.0000)
  Significant at 10%: YES


In [6]:
# --- GATE: Group Average Treatment Effects (5 quintiles of predicted CATE) ---

gate_results = None

if obj_best is not None and S_proxy is not None:
    # Re-fit with n_rep=1 for GATE compatibility
    dml_data_gate = dml.DoubleMLData(df_primary, y_col=primary_y, d_cols=primary_d, x_cols=controls)
    ml_l_gate = make_learner(best_method_primary, n_primary, len(controls))
    ml_m_gate = make_learner(best_method_primary, n_primary, len(controls))
    obj_gate = dml.DoubleMLPLR(dml_data_gate, ml_l=ml_l_gate, ml_m=ml_m_gate, n_folds=N_FOLDS, n_rep=1)
    obj_gate.fit()
    print(f'Re-fitted {best_method_primary} with n_rep=1: coef={obj_gate.coef[0]:.4f}')

    # Group by predicted CATE quintiles
    try:
        quintiles = pd.qcut(S_proxy, q=5, labels=['Q1 (low)', 'Q2', 'Q3', 'Q4', 'Q5 (high)'])
    except ValueError:
        quintiles = pd.qcut(pd.Series(S_proxy).rank(method='first'), q=5,
                           labels=['Q1 (low)', 'Q2', 'Q3', 'Q4', 'Q5 (high)'])

    groups_df = pd.get_dummies(quintiles, prefix='group', dtype=bool)
    print(f'Quintile sizes: {pd.Series(quintiles).value_counts().sort_index().to_dict()}')

    gate_obj = obj_gate.gate(groups=groups_df)
    gate_ci_df = gate_obj.confint(level=0.95, joint=True)

    gate_groups = []
    labels_list = ['Q1 (low)', 'Q2', 'Q3', 'Q4', 'Q5 (high)']
    for i, label in enumerate(labels_list):
        if i < len(gate_ci_df):
            row = gate_ci_df.iloc[i]
            coef_val = float((row.iloc[0] + row.iloc[-1]) / 2)
            gate_groups.append({'label': label, 'coef': coef_val, 'ci_lo': float(row.iloc[0]), 'ci_hi': float(row.iloc[-1])})

    het_detected = False
    for i in range(len(gate_groups)):
        for j in range(i+1, len(gate_groups)):
            if gate_groups[i]['ci_hi'] < gate_groups[j]['ci_lo'] or gate_groups[j]['ci_hi'] < gate_groups[i]['ci_lo']:
                het_detected = True

    gate_results = {
        'method': 'GATE', 'grouping_variable': 'predicted_CATE_quintile',
        'n_groups': 5, 'groups': gate_groups, 'heterogeneity_detected': het_detected,
    }
    print('\nGATE Results:')
    for g in gate_groups:
        print(f'  {g["label"]:12s}: coef={g["coef"]:+.4f}  CI=[{g["ci_lo"]:.4f}, {g["ci_hi"]:.4f}]')
    print(f'  Heterogeneity detected: {het_detected}')
else:
    obj_gate = None
    print('WARNING: Skipping GATE.')

Re-fitted Forest with n_rep=1: coef=-0.1600
Quintile sizes: {'Q1 (low)': 13, 'Q2': 12, 'Q3': 12, 'Q4': 12, 'Q5 (high)': 12}

GATE Results:
  Q1 (low)    : coef=-1.5195  CI=[-3.3695, 0.3305]
  Q2          : coef=-0.3837  CI=[-0.5780, -0.1894]
  Q3          : coef=+0.0276  CI=[-0.3344, 0.3895]
  Q4          : coef=+0.3932  CI=[0.0844, 0.7020]
  Q5 (high)   : coef=+1.6419  CI=[-0.0458, 3.3296]
  Heterogeneity detected: True


In [7]:
# --- CLAN: Classification Analysis + Write hte_results.json ---
from scipy.stats import ttest_ind

clan_results = []
if S_proxy is not None:
    top_q = S_proxy >= np.percentile(S_proxy, 80)
    bot_q = S_proxy <= np.percentile(S_proxy, 20)
    print('=== CLAN: Most vs Least Affected ===')
    print(f'  Top quintile: {top_q.sum()} obs, Bottom quintile: {bot_q.sum()} obs')
    print(f'{"Variable":35s} {"Most":>10s} {"Least":>10s} {"Diff":>8s} {"p-val":>8s}')
    print('-' * 75)
    for var in controls:
        vals_top = df_primary.loc[top_q, var].dropna()
        vals_bot = df_primary.loc[bot_q, var].dropna()
        if len(vals_top) < 2 or len(vals_bot) < 2:
            continue
        mean_top = float(vals_top.mean())
        mean_bot = float(vals_bot.mean())
        diff = mean_top - mean_bot
        t_stat, p_val = ttest_ind(vals_top, vals_bot, equal_var=False)
        clan_results.append({
            'variable': var, 'mean_most_affected': round(mean_top, 4),
            'mean_least_affected': round(mean_bot, 4), 'difference': round(diff, 4),
            'pval': round(float(p_val), 4), 'significant_10pct': bool(p_val < 0.10),
        })
        stars = '***' if p_val < 0.01 else ('**' if p_val < 0.05 else ('*' if p_val < 0.10 else ''))
        print(f'{var:35s} {mean_top:10.4f} {mean_bot:10.4f} {diff:+8.4f} {p_val:8.4f}{stars}')

# Write hte_results.json
HTE_RESULTS = RESULTS_DIR / 'hte_results.json'
hte_output = {
    'blp': blp_results,
    'gate': gate_results,
    'clan': clan_results,
    'cate_summary': None,
}
HTE_RESULTS.write_text(json.dumps(hte_output, indent=2, default=str))
print(f'\nWrote {HTE_RESULTS}')

=== CLAN: Most vs Least Affected ===
  Top quintile: 13 obs, Bottom quintile: 13 obs
Variable                                  Most      Least     Diff    p-val
---------------------------------------------------------------------------
other_taxes                             1.0723     1.5646  -0.4923   0.4983
vatsales                               15.4290    14.5917  +0.8373   0.7030
pit2004                                35.8846    36.5385  -0.6538   0.8948
lnpayments2004                          3.2324     3.4515  -0.2191   0.4846
lngdppc2003                             8.5109     8.5514  -0.0405   0.9415
propertyrights                         56.1538    56.1538  +0.0000   1.0000
sb_proc2004                             8.2308     9.6154  -1.3846   0.3394
emplo_i2004                            36.5385    40.1538  -3.6154   0.6045
freedomtotradeinternationally           6.8692     7.3769  -0.5077   0.2142
seign2004                               5.6762     5.6336  +0.0426   0.9767
inf

In [8]:
## Revision Round 1
# Fix CLAN label inversion (E1 from synthesis report).
# S_proxy is the CATE proxy: more negative = more harmed by treatment.
# bot_q (lowest S_proxy, <= 20th percentile) = most negatively affected.
# top_q (highest S_proxy, >= 80th percentile) = least negatively affected.
# The original code incorrectly mapped top_q -> "most affected" and
# bot_q -> "least affected". We swap the labels here and rewrite the JSON.

if clan_results:
    for entry in clan_results:
        # Swap the two mean columns
        old_most = entry['mean_most_affected']
        old_least = entry['mean_least_affected']
        entry['mean_most_affected'] = old_least
        entry['mean_least_affected'] = old_most
        # Difference should be most_affected minus least_affected
        entry['difference'] = round(entry['mean_most_affected'] - entry['mean_least_affected'], 4)
    print('=== CLAN labels corrected (Revision Round 1) ===')
    print('bot_q (lowest CATE) -> most affected; top_q (highest CATE) -> least affected')
    print(f'{"Variable":35s} {"Most":>10s} {"Least":>10s} {"Diff":>8s} {"p-val":>8s}')
    print('-' * 75)
    for c in clan_results:
        stars = '***' if c['pval'] < 0.01 else ('**' if c['pval'] < 0.05 else ('*' if c['pval'] < 0.10 else ''))
        print(f'{c["variable"]:35s} {c["mean_most_affected"]:10.4f} {c["mean_least_affected"]:10.4f} {c["difference"]:+8.4f} {c["pval"]:8.4f}{stars}')

    # Rewrite hte_results.json with corrected labels
    HTE_RESULTS = RESULTS_DIR / 'hte_results.json'
    hte_output = {
        'blp': blp_results,
        'gate': gate_results,
        'clan': clan_results,
        'cate_summary': None,
    }
    HTE_RESULTS.write_text(json.dumps(hte_output, indent=2, default=str))
    print(f'\nRewrote {HTE_RESULTS} with corrected CLAN labels')

=== CLAN labels corrected (Revision Round 1) ===
bot_q (lowest CATE) -> most affected; top_q (highest CATE) -> least affected
Variable                                  Most      Least     Diff    p-val
---------------------------------------------------------------------------
other_taxes                             1.5646     1.0723  +0.4923   0.4983
vatsales                               14.5917    15.4290  -0.8373   0.7030
pit2004                                36.5385    35.8846  +0.6539   0.8948
lnpayments2004                          3.4515     3.2324  +0.2191   0.4846
lngdppc2003                             8.5514     8.5109  +0.0405   0.9415
propertyrights                         56.1538    56.1538  +0.0000   1.0000
sb_proc2004                             9.6154     8.2308  +1.3846   0.3394
emplo_i2004                            40.1538    36.5385  +3.6153   0.6045
freedomtotradeinternationally           7.3769     6.8692  +0.5077   0.2142
seign2004                             

In [9]:
# --- Causal Forest (CausalForestDML) ---

CF_RESULTS_PATH = RESULTS_DIR / 'causal_forest_results.json'
cf_results = None

if ECONML_AVAILABLE:
    print('=== Causal Forest (CausalForestDML) ===')
    X_cf = df_primary[controls].values
    Y_cf = df_primary[primary_y].values
    T_cf = df_primary[primary_d].values
    n_cf = len(Y_cf)

    scaler = StandardScaler()
    X_cf_scaled = scaler.fit_transform(X_cf)

    cf_dml = CausalForestDML(
        model_y=RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42),
        model_t=RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42),
        n_estimators=1000,
        min_samples_leaf=max(5, n_cf // 10),
        max_depth=None, honest=True, inference=True,
        criterion='mse', cv=2, random_state=42,
    )
    cf_dml.fit(Y_cf, T_cf, X=X_cf_scaled, W=X_cf_scaled)

    # ATE
    ate_inf = cf_dml.ate_inference(X=X_cf_scaled)
    cf_ate = float(ate_inf.point_estimate)
    cf_ate_se = float(ate_inf.stderr)
    cf_ate_ci = ate_inf.conf_int(alpha=0.05)
    cf_ate_ci_lo = float(cf_ate_ci[0][0]) if hasattr(cf_ate_ci[0], '__len__') else float(cf_ate_ci[0])
    cf_ate_ci_hi = float(cf_ate_ci[1][0]) if hasattr(cf_ate_ci[1], '__len__') else float(cf_ate_ci[1])

    # CATE per obs
    cate_pred = cf_dml.effect(X_cf_scaled).flatten()
    cate_inf = cf_dml.effect_inference(X_cf_scaled)
    cate_ci = cate_inf.conf_int(alpha=0.05)

    # SE sanity check
    ate_ci_width = cf_ate_ci_hi - cf_ate_ci_lo
    if hasattr(cate_ci, 'shape') and len(cate_ci) == 2:
        mean_cate_ci_width = float(np.mean(cate_ci[1] - cate_ci[0]))
    else:
        mean_cate_ci_width = float(np.mean(cate_ci[:, 1] - cate_ci[:, 0]))
    ratio = mean_cate_ci_width / ate_ci_width if ate_ci_width > 0 else float('inf')
    print(f'  SE sanity: ATE CI width={ate_ci_width:.4f}, mean CATE CI width={mean_cate_ci_width:.4f}, ratio={ratio:.1f}')

    # Feature importances
    feat_imp_raw = cf_dml.feature_importances_
    feat_imp_sum = feat_imp_raw.sum()
    feat_imp_norm = feat_imp_raw / feat_imp_sum if feat_imp_sum > 0 else feat_imp_raw
    feat_imp_dict = {controls[i]: round(float(feat_imp_norm[i]), 4) for i in range(len(controls))}

    # CATE summary
    pct_neg = float(np.mean(cate_pred < 0))
    if hasattr(cate_ci, 'shape') and len(cate_ci) == 2:
        pct_sig = float(np.mean((cate_ci[0] > 0) | (cate_ci[1] < 0)))
    else:
        pct_sig = float(np.mean((cate_ci[:, 0] > 0) | (cate_ci[:, 1] < 0)))

    # Above/below median
    median_cate = float(np.median(cate_pred))
    above_mask = cate_pred >= median_cate
    ate_above = cf_dml.ate_inference(X=X_cf_scaled[above_mask])
    ate_below = cf_dml.ate_inference(X=X_cf_scaled[~above_mask])

    dml_best_coef = primary_spec['learners']['Best']['coef'] if primary_spec and 'Best' in primary_spec['learners'] else None
    sign_vs_pub = False
    sign_vs_dml = False
    if primary_spec and primary_spec['ols_coef'] is not None and primary_spec['ols_coef'] != 0:
        sign_vs_pub = bool(np.sign(cf_ate) != np.sign(primary_spec['ols_coef']))
    if dml_best_coef is not None and dml_best_coef != 0:
        sign_vs_dml = bool(np.sign(cf_ate) != np.sign(dml_best_coef))

    cf_results = {
        'method': 'CausalForestDML', 'n_obs': n_cf, 'n_estimators': 1000, 'honest': True,
        'ate': cf_ate, 'ate_se': cf_ate_se, 'ate_ci_lo': cf_ate_ci_lo, 'ate_ci_hi': cf_ate_ci_hi,
        'ate_above_median': float(ate_above.point_estimate),
        'ate_below_median': float(ate_below.point_estimate),
        'cate_summary': {
            'mean': round(float(np.mean(cate_pred)), 4), 'sd': round(float(np.std(cate_pred)), 4),
            'min': round(float(np.min(cate_pred)), 4), 'max': round(float(np.max(cate_pred)), 4),
            'pct_negative': round(pct_neg, 4), 'pct_significant': round(pct_sig, 4),
        },
        'feature_importances': dict(sorted(feat_imp_dict.items(), key=lambda x: -x[1])),
        'sign_change_vs_published': sign_vs_pub, 'sign_change_vs_dml': sign_vs_dml,
    }
    CF_RESULTS_PATH.write_text(json.dumps(cf_results, indent=2, default=str))

    # Update HTE with CATE summary
    hte_output['cate_summary'] = cf_results['cate_summary']
    HTE_RESULTS.write_text(json.dumps(hte_output, indent=2, default=str))

    print(f'\nCausal Forest ATE: {cf_ate:.4f} (SE={cf_ate_se:.4f}, CI=[{cf_ate_ci_lo:.4f}, {cf_ate_ci_hi:.4f}])')
    print(f'  Above-median: {float(ate_above.point_estimate):.4f}, Below-median: {float(ate_below.point_estimate):.4f}')
    print(f'  CATE: mean={np.mean(cate_pred):.4f}, sd={np.std(cate_pred):.4f}')
    print(f'  % negative: {pct_neg:.1%}, % significant: {pct_sig:.1%}')
    print(f'  Top features: {dict(list(cf_results["feature_importances"].items())[:5])}')
    print(f'Wrote {CF_RESULTS_PATH}')
else:
    print('econml not available. Skipping Causal Forest.')

econml not available. Skipping Causal Forest.


In [10]:
# --- Forest plot: coefficient comparison across all 4 panels ---

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, y_var in enumerate(outcomes):
    ax = axes[idx]
    spec_res = None
    for s in all_dml_results:
        if s['outcome']['variable'] == y_var and s['treatment']['variable'] == 'effective_5yr':
            spec_res = s
            break
    if spec_res is None:
        ax.set_title(panel_labels.get(y_var, y_var) + ' - no results')
        continue

    labels, coefs, ci_los, ci_his, colors = [], [], [], [], []

    # OLS
    if spec_res['ols_coef'] is not None:
        labels.append('OLS')
        coefs.append(spec_res['ols_coef'])
        ols_se = spec_res['ols_se'] or 0
        ci_los.append(spec_res['ols_coef'] - 1.96 * ols_se)
        ci_his.append(spec_res['ols_coef'] + 1.96 * ols_se)
        colors.append('#888888')

    # DML methods
    display_order = ['Lasso', 'ElasticNet', 'DecisionTree', 'Boosting', 'Forest', 'NeuralNet', 'Ensemble', 'Best']
    display_names = {'Lasso': 'Lasso', 'ElasticNet': 'Elastic Net', 'DecisionTree': 'Dec. Tree',
                     'Boosting': 'Boosting', 'Forest': 'Random Forest', 'NeuralNet': 'Neural Net',
                     'Ensemble': 'Ensemble', 'Best': 'Best'}
    for m in display_order:
        lr = spec_res['learners'].get(m, {})
        if 'coef' in lr:
            labels.append(display_names.get(m, m))
            coefs.append(lr['coef'])
            ci_los.append(lr.get('ci_lo', lr['coef']))
            ci_his.append(lr.get('ci_hi', lr['coef']))
            colors.append('#2ca02c' if m in ('Best', 'Ensemble') else '#1f77b4')

    # Causal Forest (primary panel only)
    if y_var == outcome and cf_results is not None:
        labels.append('Causal Forest')
        coefs.append(cf_results['ate'])
        ci_los.append(cf_results['ate_ci_lo'])
        ci_his.append(cf_results['ate_ci_hi'])
        colors.append('#d62728')

    y_pos = np.arange(len(labels))
    xerr_lo = [c - lo for c, lo in zip(coefs, ci_los)]
    xerr_hi = [hi - c for c, hi in zip(coefs, ci_his)]

    for i, (c, color) in enumerate(zip(coefs, colors)):
        ax.errorbar(c, i, xerr=[[coefs[i] - ci_los[i]], [ci_his[i] - coefs[i]]],
                    fmt='o', capsize=4, capthick=1.5, markersize=7,
                    color=color, ecolor=color, elinewidth=1.5, zorder=3)

    ax.axvline(x=0, color='grey', linestyle='--', linewidth=0.8, alpha=0.7)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('Coefficient (95% CI)', fontsize=9)
    ax.set_title(f'{panel_labels.get(y_var, y_var)} (N={spec_res["n_obs"]})', fontsize=11, fontweight='bold')

fig.suptitle('DML Estimates: Effect of Corporate Taxes (effective_5yr)', fontsize=13, fontweight='bold', y=1.01)
fig.tight_layout()
for ext in ['pdf', 'png']:
    fig.savefig(str(FIGURES_DIR / f'forest_plot.{ext}'), dpi=200, bbox_inches='tight')
plt.close(fig)
print(f'Saved forest_plot.pdf/.png to {FIGURES_DIR}')

Saved forest_plot.pdf/.png to C:\Users\qgallea\Dropbox\work\claude_code\recast\papers\04_djankov_et_al_taxes\paper\figures


In [11]:
# --- B&N-style LaTeX table + GATE plot + GATE/CLAN tables + CATE/feature plots ---

# ===== 1. B&N LaTeX comparison table =====
def fmt_coef(val, pval=None):
    if val is None: return '---'
    stars = ''
    if pval is not None:
        if pval < 0.01: stars = '$^{***}$'
        elif pval < 0.05: stars = '$^{**}$'
        elif pval < 0.10: stars = '$^{*}$'
    return f'{val:.3f}{stars}'

def fmt_se(val):
    if val is None: return ''
    return f'({val:.3f})'

col_methods = ['Lasso', 'DecisionTree', 'Boosting', 'Forest', 'NeuralNet', 'Ensemble', 'Best']
col_headers = ['Lasso', 'Tree', 'Boosting', 'Forest', 'NNet', 'Ensemble', 'Best', 'OLS']
treatment_display = {'statutory': 'Statutory rate', 'effective': 'Effective rate', 'effective_5yr': 'Five-year eff.'}

tex = []
tex.append(r'\begin{table}[htbp]')
tex.append(r'\centering')
tex.append(r'\caption{The effect of corporate taxes on investment and entrepreneurship: DML estimates}')
tex.append(r'\label{tab:dml_comparison}')
tex.append(r'\small')
tex.append(r'\begin{tabular}{lcccccccc}')
tex.append(r'\toprule')
tex.append(' & ' + ' & '.join([f'({i+1})' for i in range(8)]) + r' \\')
tex.append(' & ' + ' & '.join(col_headers) + r' \\')
tex.append(r'\midrule')

for y_var in outcomes:
    panel = panel_labels.get(y_var, y_var)
    tex.append(r'\multicolumn{9}{l}{\textit{' + panel + r'}} \\')
    for d_var in treatments:
        spec_res = None
        for s in all_dml_results:
            if s['outcome']['variable'] == y_var and s['treatment']['variable'] == d_var:
                spec_res = s
                break
        if spec_res is None:
            continue
        cells_c = [fmt_coef(spec_res['learners'].get(m, {}).get('coef'), spec_res['learners'].get(m, {}).get('pval')) for m in col_methods]
        ols_r = get_ols_result(y_var, d_var)
        cells_c.append(fmt_coef(spec_res.get('ols_coef'), ols_r.get('p_value') if ols_r else None))
        tex.append(f'{treatment_display.get(d_var, d_var)} & {" & ".join(cells_c)} \\\\')
        cells_s = [fmt_se(spec_res['learners'].get(m, {}).get('se')) for m in col_methods]
        cells_s.append(fmt_se(spec_res.get('ols_se')))
        tex.append(f' & {" & ".join(cells_s)} \\\\')
    n_str = str(spec_res['n_obs']) if spec_res else ''
    tex.append(f'Observations & {" & ".join([n_str]*8)} \\\\')
    tex.append(f'Controls & {" & ".join([str(len(controls))]*8)} \\\\')
    tex.append(r'\midrule')

tex[-1] = r'\bottomrule'
tex.append(r'\end{tabular}')
tex.append(r'\begin{minipage}{0.95\textwidth}')
tex.append(r'\vspace{2mm}\footnotesize')
tex.append(f'\\textit{{Notes:}} DML estimates with K={N_FOLDS} folds and {N_REP} repetitions. ')
tex.append(r'Median coefficients reported with B\&N adjusted standard errors in parentheses. ')
tex.append(r'Best learner selected by lowest nuisance MSE. ')
tex.append(r'$^{*}$ p$<$0.10, $^{**}$ p$<$0.05, $^{***}$ p$<$0.01.')
tex.append(r'\end{minipage}')
tex.append(r'\end{table}')

(TABLES_DIR / 'table_dml.tex').write_text('\n'.join(tex))
print(f'Wrote table_dml.tex')

# ===== 2. GATE plot =====
if gate_results and gate_results.get('groups'):
    groups = gate_results['groups']
    fig, ax = plt.subplots(figsize=(8, 4))
    labels_g = [g['label'] for g in groups]
    coefs_g = [g['coef'] for g in groups]
    ci_lo_g = [g['ci_lo'] for g in groups]
    ci_hi_g = [g['ci_hi'] for g in groups]
    y_pos = np.arange(len(labels_g))
    for i in range(len(groups)):
        ax.errorbar(coefs_g[i], i, xerr=[[coefs_g[i]-ci_lo_g[i]], [ci_hi_g[i]-coefs_g[i]]],
                    fmt='o', capsize=5, capthick=1.5, markersize=8, color='#1f77b4', ecolor='#1f77b4')
    ax.axvline(x=0, color='grey', linestyle='--', linewidth=0.8, alpha=0.7)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels_g, fontsize=10)
    ax.invert_yaxis()
    ax.set_xlabel('GATE Coefficient (jointly valid 95% CI)', fontsize=10)
    ax.set_title('Group Average Treatment Effects\n(grouped by predicted CATE quintile)', fontsize=12, fontweight='bold')
    fig.tight_layout()
    for ext in ['pdf', 'png']:
        fig.savefig(str(FIGURES_DIR / f'gate_plot.{ext}'), dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved gate_plot.pdf/.png')

    # GATE LaTeX table
    gt = [r'\begin{table}[h]', r'\centering', r'\caption{Group Average Treatment Effects (GATE)}',
          r'\label{tab:gate}', r'\begin{tabular}{lccc}', r'\toprule',
          r'Quintile & Coef. & 95\% CI (joint) \\', r'\midrule']
    for g in groups:
        gt.append(f'{g["label"]} & {g["coef"]:.3f} & [{g["ci_lo"]:.3f}, {g["ci_hi"]:.3f}] \\\\')
    gt.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    (TABLES_DIR / 'table_gate.tex').write_text('\n'.join(gt))
    print(f'Wrote table_gate.tex')

# ===== 3. CLAN LaTeX table =====
if clan_results:
    ct = [r'\begin{table}[h]', r'\centering',
          r'\caption{Classification Analysis (CLAN): Most vs.\ Least Affected}',
          r'\label{tab:clan}', r'\begin{tabular}{lcccc}', r'\toprule',
          r'Variable & Most Affected & Least Affected & Difference & p-value \\', r'\midrule']
    for c in clan_results:
        vname = c['variable'].replace('_', r'\_')
        stars = '$^{***}$' if c['pval'] < 0.01 else ('$^{**}$' if c['pval'] < 0.05 else ('$^{*}$' if c['pval'] < 0.10 else ''))
        ct.append(f'{vname} & {c["mean_most_affected"]:.3f} & {c["mean_least_affected"]:.3f} & {c["difference"]:.3f} & {c["pval"]:.3f}{stars} \\\\')
    ct.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    (TABLES_DIR / 'table_clan.tex').write_text('\n'.join(ct))
    print(f'Wrote table_clan.tex')

# ===== 4. CATE histogram (from causal forest) =====
if cf_results is not None and ECONML_AVAILABLE:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(cate_pred, bins=20, alpha=0.7, color='#1f77b4', edgecolor='black', density=True)
    ax.axvline(x=0, color='red', linestyle='--', linewidth=1.5, label='Zero')
    ax.axvline(x=cf_results['ate'], color='green', linestyle='-', linewidth=2, label=f'ATE={cf_results["ate"]:.3f}')
    ax.set_xlabel('Conditional Average Treatment Effect (CATE)', fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.set_title('Distribution of Individual-Level CATE\n(CausalForestDML)', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    fig.tight_layout()
    for ext in ['pdf', 'png']:
        fig.savefig(str(FIGURES_DIR / f'cate_histogram.{ext}'), dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved cate_histogram.pdf/.png')

    # Feature importance plot
    fi = cf_results['feature_importances']
    fi_sorted = sorted(fi.items(), key=lambda x: x[1], reverse=True)
    fig, ax = plt.subplots(figsize=(8, 5))
    names = [x[0] for x in fi_sorted]
    vals = [x[1] for x in fi_sorted]
    ax.barh(range(len(names)), vals, color='#2ca02c', edgecolor='black')
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('Feature Importance', fontsize=11)
    ax.set_title('Causal Forest: Heterogeneity Drivers', fontsize=12, fontweight='bold')
    fig.tight_layout()
    for ext in ['pdf', 'png']:
        fig.savefig(str(FIGURES_DIR / f'feature_importance.{ext}'), dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved feature_importance.pdf/.png')

print('\n=== Stage 04 complete ===')

Wrote table_dml.tex
Saved gate_plot.pdf/.png
Wrote table_gate.tex
Wrote table_clan.tex

=== Stage 04 complete ===
